# 01/데이터 준비 & grounded 합성: 도메인 QA / instruction

**요약**: 공개 시드 데이터를 소량 샘플링한 뒤 Bedrock Converse로 grounded 합성 데이터를 생성하여 학습용 JSONL을 구성합니다.

**목적**: 적은 수의 시드로 파이프라인을 빠르게 검증하고, 도메인에 맞춘 합성 데이터로 품질을 끌어올리는 production 패턴을 실제로 따라 해 봅니다.

**배경**: 라벨 데이터가 부족하면 파인튜닝 자체가 어렵습니다. 그렇다고 근거 없이 합성하면 hallucination이 섞여 들어가므로, seed에 grounded된 데이터만 생성하고 critique 단계로 저품질 샘플을 걸러 냅니다.

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib
from common import config, gemma_format; importlib.reload(config)
from common.synth import bedrock_synth as bs
import importlib, track_data as td; importlib.reload(td)
TRACK = config.TRACKS['domain_qa']   # 이 트랙의 설정(시드 데이터셋 등)
NUM_SEED = 8 if config.is_dry_run() else config.NUM_SEED_SAMPLES
NUM_SYNTH = 6 if config.is_dry_run() else config.NUM_SYNTHETIC
print(f'track={TRACK.name}, seed_dataset={TRACK.seed_dataset}')
print(f'seed={NUM_SEED}, synthetic={NUM_SYNTH}, dry_run={config.is_dry_run()}')

## 1. 시드 로드 & 특성 확인
합성 데이터의 품질은 시드 데이터의 특성을 얼마나 잘 반영하느냐에 달려 있습니다. 먼저 공개 시드를 로드해 입력/출력의 형태와 길이, 어투 등을 눈으로 확인하고, 이 특성이 이후 합성/학습 단계로 이어지도록 합니다.

**시드 데이터셋**: [`databricks/databricks-dolly-15k`](https://huggingface.co/datasets/databricks/databricks-dolly-15k) (CC-BY-SA-3.0, ungated). 사람이 작성한 instruction-following 데이터셋입니다.
- **원본 포맷**: `instruction` + `context`(선택) + `response` + `category`(예: open_qa, closed_qa, summarization).
- **이 트랙의 파싱**: `instruction`과 `context`를 `input`, `response`를 `output`으로 변환합니다.
- **성공 기준**: Bedrock LLM-judge(correctness/helpfulness/groundedness) + ROUGE-L proxy.
- **라이선스**: CC-BY-SA의 share-alike 의무가 파생물에 적용됩니다.

**원본 row 예시** (raw):
```text
instruction: When did Virgin Australia start operating?
context:     Virgin Australia... commenced services on 31 August 2000... (선택 필드, 없을 수도)
response:    Virgin Australia commenced services on 31 August 2000 as Virgin Blue...
category:    closed_qa
```
파싱 후: `input`=instruction(+context), `output`=response

아래 셀은 원본 데이터셋을 이 트랙의 어댑터(`track_data.py`)로 파싱해 표준 `{"input", "output"}` 형태로 만든 뒤, 첫 샘플을 출력합니다. `load_seed_examples`가 원본 row를 어떻게 파싱하는지는 `track_data.py`를 참고하세요.

In [ ]:
seeds = td.load_seed_examples(NUM_SEED, token=config.get_hf_token())
print(f'Parsed seeds: {len(seeds)}  (dataset: {TRACK.seed_dataset})')
print('--- sample input  ---\n', seeds[0]['input'][:400])
print('--- sample output ---\n', seeds[0]['output'][:400])

## 2. grounded 합성 (Bedrock Converse + critique/refine)
시드 텍스트를 근거(grounding)로 제시하고 Bedrock Converse에 합성 데이터를 요청한 뒤, 생성 결과를 다시 모델로 평가(critique)해 groundedness와 relevance 점수가 기준에 미치지 못하는 샘플을 걸러 내는 critique/refine 루프를 돕니다. 이렇게 하면 시드에서 벗어난 hallucination을 억제할 수 있습니다.
생성 배치와 critique는 Bedrock 호출이 I/O 바운드이므로 `max_workers`만큼 **병렬**로 처리되고, **tqdm 진행바**로 채택된 예시 수가 실시간으로 표시됩니다(throttling이 나면 `max_workers`를 낮추세요).
Bedrock 모델 ID는 env `BEDROCK_CLAUDE_MODEL_ID`로 지정하며, 호출량 기준으로 과금됩니다.

> 대안: 활발히 유지보수되는 오픈 라이브러리(Kiln native Bedrock / Bespoke Curator via LiteLLM)는 `common/synth/README.md`를 참고하세요. distilabel은 유지보수가 정체되어 사용하지 않습니다.

In [ ]:
assert config.BEDROCK_CLAUDE_MODEL_ID and 'claude' in config.BEDROCK_CLAUDE_MODEL_ID, \
    'BEDROCK_CLAUDE_MODEL_ID 를 inference-profile ID로 세팅하세요 (예: global.anthropic.claude-sonnet-5)'

# 실시간 미리보기: 채택되는 예시의 처음 몇 개를 생성 중에 바로 출력
PREVIEW_N = 3
_shown = {'n': 0}
def _preview(done, total):
    if _shown['n'] < PREVIEW_N and synth_ref and len(synth_ref[0]) >= done:
        ex = synth_ref[0][done - 1]
        u = next((m['content'] for m in ex.messages if m['role'] == 'user'), '')
        a = next((m['content'] for m in reversed(ex.messages) if m['role'] == 'assistant'), '')
        print(f"\n[preview #{done}] g/r={ex.groundedness}/{ex.relevance}\n  in : {u[:120]}\n  out: {a[:120]}")
        _shown['n'] += 1
synth_ref = []   # generate_grounded가 채우는 리스트 참조(progress 시점에 접근)
synth = bs.generate_grounded(
    task_instruction=td.TASK_INSTRUCTION,
    seed_texts=td.seed_texts_for_synth(seeds),
    n_total=NUM_SYNTH,
    model_id=config.BEDROCK_CLAUDE_MODEL_ID,
    region=config.BEDROCK_REGION,
    to_messages=td.to_messages,
    max_batches=3 if config.is_dry_run() else None,
    max_workers=config.SYNTH_MAX_WORKERS,   # 동시 Bedrock 호출 수 (config/env, throttling 시 낮추기)
    accepted_ref=synth_ref,                  # 실시간 미리보기용 참조
    progress_cb=_preview,
)
print(f'\nAccepted synthetic examples: {len(synth)}')

## 3. seed vs 합성 EDA (분포/다양성/품질 점검)
학습에 넣기 전에 합성 데이터를 정량 점검합니다. 여기서 30초 쓰면, 몇 시간짜리 학습을 버리는 일을 막을 수 있습니다.

| 점검 | 무엇을 보나 | 문제면 무엇을 바꾸나 |
|---|---|---|
| 길이/중복 | 건수, 문자 길이 분포, 완전중복률 | 분포 이탈 → 생성 프롬프트 |
| **토큰 길이** | 실제 토크나이저 기준 + 절단 위험 | **`max_seq_length` 결정** |
| **근사중복** | 합성끼리 닮음 + **seed 표절** | temperature↑ / **평가 누출 차단** |
| **어휘 다양성** | distinct-1/2, 시작 3-gram 편중 | 도입부 템플릿 고착 해소 |
| 클래스 균형 | 라벨 분포/소수 클래스 소실 | 소수 클래스 추가 생성 |

**토큰 길이가 특히 중요합니다.** 학습이 자르는 단위는 문자가 아니라 토큰이고, 한국어처럼 영어가 아닌 텍스트는 문자당 토큰 수가 영어의 몇 배입니다. 문자 길이로는 안전해 보여도 토큰으로는 `max_seq_length`를 넘어 **정답 뒷부분이 잘린 채 학습**될 수 있습니다: 그러면 모델은 '끝나지 않는 출력'을 정답으로 배웁니다.
**seed 표절도 놓치기 쉽습니다.** 합성이 seed를 거의 그대로 베끼면 증강 효과가 없고, 그 seed를 held-out으로 쓰면 평가 점수가 부풀려집니다(누출).

각 점검은 문제를 찾으면 주의: 와 함께 **구체적 조치**를 함께 출력합니다.

In [ ]:
from common.synth import eda
from transformers import AutoTokenizer
# 토큰 길이를 실제 학습 토크나이저로 재려면 tokenizer를 넘깁니다(권장: max_seq_length 결정에 직결).
_tok = AutoTokenizer.from_pretrained(config.DEFAULT_MODEL_ID, token=config.get_hf_token())
stats = eda.quick_report(
    seeds, synth,
    tokenizer=_tok,
    max_seq_length=1024,   # 02 학습에서 쓰는 값과 동일하게
    plot=True,
)


> 개별 점검만 다시 돌리고 싶다면: `eda.compare`/`eda.token_length_report`/`eda.near_duplicate_report`/`eda.lexical_diversity`/`eda.label_balance`/`eda.output_validity` (구현: `common/synth/eda.py`).
> 근사중복은 O(n²) 비교라 기본 400건까지만 샘플링합니다: 전량을 보려면 `sample=` 를 키우세요.

## 4. 시드 + 합성 병합 → 학습 JSONL (messages 포맷)
원본 시드와 검증을 통과한 합성 데이터를 하나로 합쳐 학습셋을 구성합니다. 각 예시는 대화형 `messages` 포맷으로 저장하는데, 이는 이후 학습 단계에서 chat template이 자동 적용되도록 하기 위한 표준 형태입니다. 완성된 경로는 `%store`로 저장해 학습 노트북에서 그대로 사용합니다.

In [ ]:
import os, json
os.makedirs('data', exist_ok=True)
all_msgs = [td.to_messages(s) for s in seeds] + [ex.messages for ex in synth]
train_path = 'data/train.jsonl'
with open(train_path, 'w', encoding='utf-8') as f:
    for m in all_msgs:
        f.write(json.dumps({'messages': m}, ensure_ascii=False) + '\n')
print(f'Training set: {len(all_msgs)} examples -> {train_path}')
# (train_path는 %store 안 함: 02는 이 트랙의 로컬 data/train.jsonl을 직접 사용해 트랙 오염 방지)

## 5. 포맷 검증: Gemma chat template 적용 미리보기
학습에 넘기기 전에, 토크나이저의 `apply_chat_template`을 직접 적용해 보고 `<start_of_turn>` 같은 Gemma 대화 마커가 의도대로 조립되는지 확인합니다. 학습 시점에 자동으로 적용되는 것과 동일한 변환을 미리 눈으로 검증해 두면 포맷 불일치로 인한 학습 실패를 예방할 수 있습니다.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(config.DEFAULT_MODEL_ID, token=config.get_hf_token())
print(tok.apply_chat_template(all_msgs[0], tokenize=False)[:600])

데이터 준비가 끝났습니다. 다음은 **02_train_sft_sagemaker.ipynb**로 이어집니다. (합성 데이터를 대량으로 생성하면 Bedrock 호출 비용이 늘어나므로 주의하세요.)